# Session Selection — DANDI_000688

Scan all DANDI_000688 sessions (2013-2016) in the local zarr store.
Extract metrics: electrode count, trial count, duration, firing rate, velocity symmetry.
Filter by quality criteria and rank candidates for analysis.

## Candidate Selection Criteria
- **n_trials ≥ 120** — enough trials for robust K-Fold CV
- **n_active_electrodes ≥ 150** — sufficient neural signal
- **symmetry_ratio ≥ 0.7** — vx and vy have comparable variability (min/max std ratio)

In [ ]:
import numpy as np
import pandas as pd
import zarr
from pathlib import Path

ZARR_PATH = "/home/camilavelasquez/Documents/Datasets/Combined_Motor_Datasets_V9.zarr"
BIN_SIZE_MS = 50

print(f"Opening zarr store: {ZARR_PATH}")

## List All Sessions

In [ ]:
store = zarr.DirectoryStore(ZARR_PATH)
root = zarr.group(store=store, overwrite=False)

all_sessions = list(root.keys())
print(f"Total sessions in store: {len(all_sessions)}")
print(f"Sample keys: {all_sessions[:5]}")

## Filter to DANDI_000688 Years (2013-2016)

In [ ]:
# Sessions are keys like '20131203', '20140215', etc.
# DANDI_000688 sessions are from 2013-2016
dandi_sessions = [s for s in all_sessions if s.startswith(('2013', '2014', '2015', '2016'))]

print(f"DANDI_000688 candidate sessions (2013-2016): {len(dandi_sessions)}")
print(f"Years present: {sorted(set(s[:4] for s in dandi_sessions))}")

## Extract Metrics for Each Session

For each session, compute:
- **n_electrodes**: number of spike channels
- **n_bins**: total time bins (50 ms each)
- **duration_min**: session duration in minutes
- **n_trials**: unique positive trial IDs (excluding 0)
- **n_active_electrodes**: channels with mean FR > 0.5 Hz
- **mean_fr**: mean firing rate across active channels (Hz)
- **vx_std, vy_std**: velocity component std (after reshaping to (n_bins, 2))
- **symmetry_ratio**: min/max of (vx_std, vy_std) — closer to 1.0 is better

In [ ]:
results = []

for session_key in dandi_sessions:
    try:
        session = root[session_key]
        
        # Extract data
        spikes = session['spikes'][:]  # (n_el, n_time)
        velocity = session['velocity'][:]  # (n_time, 2) or (2, n_time)
        trial_ids = session['trial_id'][:]  # (n_time,)
        
        n_electrodes = spikes.shape[0]
        n_time = spikes.shape[1]
        n_bins = n_time // BIN_SIZE_MS
        duration_min = n_time / 30000.0  # assuming 30 kHz
        
        # Compute binned spike counts
        spikes_binned = spikes[:, :n_bins * BIN_SIZE_MS].reshape(n_electrodes, n_bins, BIN_SIZE_MS).sum(axis=2)
        mean_fr_per_el = spikes_binned.mean() / (BIN_SIZE_MS / 1000.0)  # Hz
        active_els = (spikes_binned.mean(axis=1) / (BIN_SIZE_MS / 1000.0) > 0.5).sum()
        
        # Count unique positive trial IDs
        unique_trials = np.unique(trial_ids[trial_ids > 0])
        n_trials = len(unique_trials)
        
        # Velocity: handle shape (n_time, 2) or (2, n_time)
        if velocity.shape[0] == 2 and velocity.shape[1] != 2:
            velocity = velocity.T  # (2, n_time) → (n_time, 2)
        
        # Reshape and compute stats
        vel_binned = velocity[:n_bins * BIN_SIZE_MS].reshape(n_bins, BIN_SIZE_MS, 2).mean(axis=1)
        vx_std = vel_binned[:, 0].std()
        vy_std = vel_binned[:, 1].std()
        symmetry = min(vx_std, vy_std) / max(vx_std, vy_std) if max(vx_std, vy_std) > 0 else 0.0
        
        results.append({
            'session': session_key,
            'n_electrodes': n_electrodes,
            'n_bins': n_bins,
            'duration_min': round(duration_min, 2),
            'n_trials': n_trials,
            'n_active_electrodes': active_els,
            'mean_fr': round(mean_fr_per_el, 2),
            'vx_std': round(vx_std, 4),
            'vy_std': round(vy_std, 4),
            'symmetry_ratio': round(symmetry, 3),
        })
        print(f'✓ {session_key}: {n_trials} trials, {n_electrodes} els, {active_els} active, sym={symmetry:.3f}')
    except Exception as e:
        print(f'✗ {session_key}: {e}')
        continue

print(f"\nSuccessfully processed {len(results)} sessions")

## Session Summary Table (sorted by n_trials)

In [ ]:
df = pd.DataFrame(results)
df = df.sort_values('n_trials', ascending=False).reset_index(drop=True)

print(f"\n=== All {len(df)} Sessions ===")
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', None):
    display(df)

## Apply Quality Filters

In [ ]:
# Quality thresholds
MIN_TRIALS = 120
MIN_ACTIVE_ELS = 150
MIN_SYMMETRY = 0.7

mask = (df['n_trials'] >= MIN_TRIALS) & \
        (df['n_active_electrodes'] >= MIN_ACTIVE_ELS) & \
        (df['symmetry_ratio'] >= MIN_SYMMETRY)

candidates = df[mask].reset_index(drop=True)

print(f"Sessions passing filters:")
print(f"  n_trials >= {MIN_TRIALS}: {(df['n_trials'] >= MIN_TRIALS).sum()}")
print(f"  n_active_electrodes >= {MIN_ACTIVE_ELS}: {(df['n_active_electrodes'] >= MIN_ACTIVE_ELS).sum()}")
print(f"  symmetry_ratio >= {MIN_SYMMETRY}: {(df['symmetry_ratio'] >= MIN_SYMMETRY).sum()}")
print(f"\nCombined (all 3 criteria): {len(candidates)} sessions\n")

with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', None):
    display(candidates)

## Top 5 Candidates (by n_trials)

In [ ]:
top5 = candidates.head(5) if len(candidates) > 0 else df.head(5)

print(f"\n=== Top 5 Sessions (by n_trials) ===")
for idx, row in top5.iterrows():
    print(f"\n[{idx+1}] {row['session']}")
    print(f"    Trials: {row['n_trials']} | Electrodes: {row['n_electrodes']} (active={row['n_active_electrodes']})")
    print(f"    Duration: {row['duration_min']} min | Mean FR: {row['mean_fr']} Hz")
    print(f"    Velocity: vx_std={row['vx_std']}, vy_std={row['vy_std']}, symmetry={row['symmetry_ratio']}")

# Recommendation
if len(candidates) > 0:
    best = candidates.iloc[0]
    print(f"\n→ RECOMMENDED: {best['session']} ({best['n_trials']} trials, {best['n_active_electrodes']} active electrodes)")
else:
    print(f"\n⚠ No sessions pass all quality filters. Relaxing criteria...")
    best = df.iloc[0]
    print(f"→ BEST AVAILABLE: {best['session']} ({best['n_trials']} trials, {best['n_active_electrodes']} active electrodes)")